# 07.05 · Rerankers

**Chapter:** Embeddings & Vector Search · **Milestone:** M3

## Goals
- [ ] Bi-encoders (embeddings) vs cross-encoders (rerankers) vs late interaction (ColBERT)
- [ ] Add a reranking stage: retrieve 30–100 cheaply, rerank to the best 5
- [ ] Measure the gain and the latency cost

## Resources
- [Sentence Transformers – Cross-encoders](https://sbert.net/docs/cross_encoder/usage/usage.html) — free
- [Pinecone – Rerankers and two-stage retrieval](https://www.pinecone.io/learn/series/rag/rerankers/) — free
- [ColBERT](https://github.com/stanford-futuredata/ColBERT) · [RAGatouille](https://github.com/AnswerDotAI/RAGatouille) — free

| Type | How | Speed | Quality |
|---|---|---|---|
| **Bi-encoder** | query and doc embedded separately, compare vectors | doc vectors precomputed → very fast | good |
| **Cross-encoder** | query + doc read *together* by one transformer → relevance score | must run per (query, doc) pair → slow | best |
| **Late interaction (ColBERT)** | per-token vectors, MaxSim matching | in between | very good |
| **LLM as reranker** | prompt an LLM to rank/score | slowest, priciest | strong, flexible |

Options: open (`cross-encoder/ms-marco-MiniLM-L6-v2`, `BAAI/bge-reranker-v2-m3`, `mixedbread-ai/mxbai-rerank-*`, Qwen3-Reranker) or API
(Cohere Rerank, Voyage, Jina).

In [ ]:
import time

import bm25s
import numpy as np
from sentence_transformers import CrossEncoder

from aieng import rag

docs = rag.corpus()
evalset = rag.eval_set()
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

## 1. A cross-encoder scores pairs

In [ ]:
query = "When was Abraham Lincoln born?"
candidates = [text for _, _, text in rag.search(query, 10)]
scores = reranker.predict([(query, c) for c in candidates])
for s, c in sorted(zip(scores, candidates), reverse=True)[:4]:
    print(f"{s:7.2f}  {c[:100]}")

## 2. Two-stage retrieval: hybrid → rerank
Re-using BM25 + dense + RRF from 07.04.

In [ ]:
bm25 = bm25s.BM25(); bm25.index(bm25s.tokenize(docs, stopwords="en", show_progress=False), show_progress=False)


def hybrid(query, depth=30):
    ids, _ = bm25.retrieve(bm25s.tokenize([query], stopwords="en", show_progress=False), k=depth, show_progress=False)
    rankings = [[int(i) for i in ids[0]], [pid for pid, _, _ in rag.search(query, depth)]]
    fused = {}
    for ranking in rankings:
        for rank, d in enumerate(ranking):
            fused[d] = fused.get(d, 0) + 1 / (61 + rank)
    return sorted(fused, key=fused.get, reverse=True)[:depth]


def rerank(query, ids, k=5):
    s = reranker.predict([(query, docs[i]) for i in ids], batch_size=32)
    return [ids[j] for j in np.argsort(-s)[:k]]


def evaluate(search_fn, k=5):
    start = time.perf_counter()
    hits = [bool(set(search_fn(r["question"])[:k]) & r["gold"]) for r in evalset]
    return np.mean(hits), (time.perf_counter() - start) * 1000 / len(evalset)


for name, fn in {
    "dense top-5": lambda q: [p for p, _, _ in rag.search(q, 5)],
    "hybrid top-5": lambda q: hybrid(q)[:5],
    "hybrid top-30 -> rerank 5": lambda q: rerank(q, hybrid(q, 30)),
}.items():
    r, ms = evaluate(fn)
    print(f"{name:28} recall@5 {r:.3f} | {ms:6.1f} ms/query")

The reranker costs tens of ms per query (on a laptop) and usually buys the biggest single quality jump in a RAG pipeline.
**Depth** is the knob: rerank more candidates = higher recall ceiling, more latency.

## 3. LLM as a reranker (sketch)
Ask an LLM for a 0–10 relevance score per passage (or a ranked list of ids). Flexible ("prefer recent, official sources"), but slow and costly;
fine for small candidate sets or offline labelling (chapter 10).

In [ ]:
from pydantic import BaseModel

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}


class Relevance(BaseModel):
    score: int  # 0-10


def llm_score(query, passage):
    r = c.chat.completions.parse(model=MODEL, temperature=0, response_format=Relevance, **EXTRA, messages=[{"role": "user", "content":
        f"Rate 0-10 how well the passage answers the question.\nQuestion: {query}\nPassage: {passage}"}])
    return r.choices[0].message.parsed.score


ids = hybrid(query, 8)
for pid, s in sorted(((i, llm_score(query, docs[i])) for i in ids), key=lambda x: -x[1])[:3]:
    print(s, docs[pid][:100])

A 1.7B model is a poor relevance judge (check whether the birth passage scored highest). LLM reranking needs a capable model —
rerun with `client("gemini")` or `client("groq")` and compare.

## Exercises
1. Sweep the rerank depth (10, 30, 100). Plot recall@5 and latency.
2. Swap in `mixedbread-ai/mxbai-rerank-xsmall-v1` or `BAAI/bge-reranker-v2-m3` (bigger, multilingual). Better? Slower?
3. Cohere has a free trial key: `uv add cohere`, then `co.rerank(model="rerank-v3.5", query=q, documents=[...], top_n=5)`. Compare with the open reranker.

In [ ]:
# your code here